
# Scenario overview

Reads a scenario's **setup** straight from the intent database -- no run
required. Complements [accounts.ipynb](./accounts.ipynb) (account detail)
and [net_worth.ipynb](./net_worth.ipynb) (simulation results): this notebook
is about what was *authored* (manifest, persons, accounts, scheduled
transactions, chains, calendar events), not what the engine produced.

Pick a scenario below, then re-run the cells below.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

from IPython.display import display
from rich.console import Console

from analysis.sim.loader import resolve_scenario_name
from analysis.sim.notebook_ui.pickers import scenario_picker
from analysis.sim.accounts import load_accounts_view, render_accounts_table
from analysis.sim.notebook_ui.scenario_view import (
    manifest_view, persons_view, scheduled_view, calendar_view, chains_view
)


DATA_DIR = Path(os.environ.get("ANALYSIS_DATA_DIR", "../data/shared/sim"))
# Same resolution the CLI uses: $ANALYSIS_SCENARIO, else the data repo's
# .current_scenario, else "current".
_default_scenario = resolve_scenario_name(
    DATA_DIR, os.environ.get("ANALYSIS_SCENARIO") or None
).name

# Pick a different scenario here, then re-run this cell and the ones below.
# CONNECTIONS.scenario is a live ScenarioConnection -- this notebook never
# needs a run, so `require_run=False`.
PICKER, CONNECTIONS = scenario_picker(DATA_DIR, _default_scenario, require_run=False)
display(PICKER)


## Manifest

The intent file's identity card -- what scenario this is, what code/data
version built it, and when.


In [ ]:
manifest_view(CONNECTIONS.scenario)


## Persons

Everyone this scenario's accounts, scheduled transactions, and lifespan
rules can reference.


In [ ]:
persons_view(CONNECTIONS.scenario)


## Accounts

Same view as [accounts.ipynb](./accounts.ipynb)'s rich table: kind, owner,
tax treatment, opening balance, and applicable growth/yield rate.


In [ ]:
rows = load_accounts_view(CONNECTIONS.scenario)
Console().print(render_accounts_table(rows, title=f"accounts ({CONNECTIONS.scenario.scenario})"))


## Chains

Liquidity chains (ADR-003 §4.7): every account is also an implicit leaf
chain (draws directly from itself, no steps); an interior chain instead
lists an ordered set of draw steps, each pulling from another chain until
its need is satisfied.


In [ ]:
chains = chains_view(CONNECTIONS.scenario)
chains


In [ ]:
# Interior chains only (the ones with actual draw steps) -- the leaf rows
# above are mostly just "every account is also a chain of its own name".
chains[chains["step"].notna()]


## Scheduled transactions

Every recurring rule, in the order the engine evaluates them (priority,
then name -- ADR-001's priority bands).


In [ ]:
scheduled_view(CONNECTIONS.scenario).style.format({"amount": "${:,.2f}"})


## Calendar events

One-time interventions (`set_amount`, retirements, benefit cuts, etc.), in
date order.


In [ ]:
calendar_view(CONNECTIONS.scenario).style.format({"amount": "${:,.2f}"})
